# CD CosMx Resection: Per-FOV LIANA Analysis

Runs LIANA bivariate ligand-receptor scores per FOV on one resection run at a time (set `sample` and `patient_sec`; shown for CD_R1_B7, Cos4).
Gaussian kernel (bandwidth 200 px, cutoff 0.1, at most 100 neighbours) within each FOV, NicheNet LR network, 100 permutations → per-FOV h5ad outputs (`Processed/liana/fov_<n>.h5ad`).
The per-FOV results of all runs are merged in `07_resection_liana_merge.ipynb`.

In [ ]:
import scanpy as sc
import pandas as pd
import os
import liana as li
import pyreadr
import gc

In [ ]:
samples = [
'Cos1_6KDC_strict18888_A5_0715_11_08_2025_14_21_12_760',                
'Cos1_6KDC_strict43563_SB_A1_091725BP6_09_10_2025_13_54_55_807',          
'Cos1_6KDC_strict43563_SB_A8_091725BP5_09_10_2025_13_41_29_437',       
'Cos1_6KDC_strictCD_R1_B4_0715_11_08_2025_14_20_23_133',     
'Cos2_6KDC_strict18888_A10_0715_Alan_data_test_11_08_2025_14_25_11_455',  
'Cos2_6KDC_strict18888_A6_0715_Alan_Test_11_08_2025_14_23_09_489',  
'Cos3_6KDC_strict11808_A4_0716_11_08_2025_14_19_48_615',                  
'Cos3_6KDC_strictCD_R1_B5_0716_11_08_2025_14_17_56_277',                
'Cos4_6KDC_strict11808_A4_0716_14_10_2025_12_34_38_727',
'Cos4_6KDC_strictCD_R1_B7_0716_11_08_2025_14_18_41_738',
'Cos5_6KDC_strict43563_SB_A11_091725BP7_09_10_2025_13_42_42_775',         
'Cos5_6KDC_strict43563_SB_A4_091725BP8_09_10_2025_13_43_50_952'  
]

In [ ]:
mapper= pd.read_csv('/path/to/scrna/cd/cell_type_category_map.csv')

In [ ]:
# Clean the names
mapper['cell_type_aucell'] = mapper['cell type'].str.replace(r'[+\-\/()]', ' ', regex=True)
mapper['cell_type_aucell'] = mapper['cell_type_aucell'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Disambiguate duplicates by appending " 1", " 2", etc.
mapper['cell_type_aucell'] = (
    mapper.groupby('cell_type_aucell').cumcount().astype(str).replace('0', '', regex=False)
    .radd(mapper['cell_type_aucell'] + ' ').str.strip()
)

In [ ]:
prefix = '/path/to/cosmx_data/cd_resection/'

In [ ]:
j = 0
for i in samples:
    df = pd.read_csv(prefix+i+'/aucell_anno.csv')
    df['label_clean'] = df['label'].str.rstrip()
    df=df.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')
    if 'SB' in i:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[2]
        comb = patient+'_'+section
    elif '11808' in i:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[1]
        machine = i.split('_')[0]
        comb = patient+'_'+section+'_'+machine
    else:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[1]
        comb = patient+'_'+section
    
    df['patient']=patient
    df['patient']=section
    df['patient_sec'] = comb
    if j == 0:
        merged = df
    else:
        merged = pd.concat([merged,df],ignore_index=True)
    j+=1

In [ ]:
# Now read it with pyreadr
lr = pyreadr.read_r("/path/to/nichenet/lr_network_human_21122021.rds")

In [ ]:
lr_df = next(iter(lr.values()))

In [ ]:
lr_df=lr_df[['from','to']]

In [ ]:
lr_df.columns = ['ligand','receptor']

In [ ]:
# run index (0-11)
sample = samples[9]

In [ ]:
patient_sec = 'CD_R1_B7'

In [ ]:
adata = sc.read_h5ad(prefix+sample+'/Processed/'+'norm_log1p.h5ad')

In [ ]:
merged_sub = merged[merged['patient_sec']==patient_sec]

In [ ]:
adata.obs['cell_type'] = merged_sub['cell type'].tolist()
adata.obs['cell_type_short'] = merged_sub['cell type short'].tolist()
adata.obs['cell_category'] = merged_sub['cell category'].tolist()
adata.obs['cell_type_general'] = merged_sub['cell type general'].tolist()

In [ ]:
base_cats = list(adata.obs['cell_category'].astype('category').cat.categories)
reordered_cats = [cat for cat in base_cats if cat != 'Cycling'] + ['Cycling']

adata.obs['cell_category'] = pd.Categorical(
    adata.obs['cell_category'],
    categories=reordered_cats,
    ordered=True
) 

In [ ]:
# Copy spatial_fov to spatial, but flip the Y-axis
coords = adata.obsm['spatial_fov'].copy()

# Flip Y coordinates
y_max = coords[:, 1].max()
y_min = coords[:, 1].min()
coords[:, 1] = y_max + y_min - coords[:, 1]  # Mirror around center

# Set as spatial
adata.obsm['spatial'] = coords

In [ ]:
fovs = adata.obs['fov'].unique().tolist()

In [ ]:
for i in fovs:
    adata_sub = adata[adata.obs['fov'] == i]
    n_cells = adata_sub.n_obs

    # Skip FOVs with too few cells
    if n_cells <= 100:  # max_neighbours is 100, so need more than that
        continue

    # Compute spatial neighbors
    li.ut.spatial_neighbors(
        adata_sub,
        bandwidth=200,
        cutoff=0.1,
        kernel='gaussian',
        set_diag=True,
        max_neighbours=100
    )

    try:
        lrdata_nn = li.mt.bivariate(
            adata_sub,
            resource=lr_df,
            local_name='cosine',
            global_name="morans",
            n_perms=100,
            mask_negatives=False,
            add_categories=True,
            nz_prop=0.05,
            use_raw=False,
            verbose=True
        )

    except ValueError as e:
        if "No features with non-zero proportions" in str(e):   # skip FOVs without expressed LR pairs
            gc.collect()
            continue
        else:
            raise

    # Build output path and create directories if needed
    out_path = (
        prefix
        + sample
        + '/Processed/liana/fov_' + i + '.h5ad'
    )
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    del lrdata_nn.uns
    del lrdata_nn.obsm

    # Subset obs safely
    cols_keep = [
        'fov', 'cell_id', 'sample', 'tissue_section',
        'cell_type', 'cell_type_short', 'cell_category', 'cell_type_general'
    ]
    cols_keep = [c for c in cols_keep if c in lrdata_nn.obs.columns]
    lrdata_nn.obs = lrdata_nn.obs[cols_keep]

    lrdata_nn.write_h5ad(out_path)
    del lrdata_nn
    gc.collect()